In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import emoji
# Natural language processing libraries
import re
import bleach
# Keras and TensorFlow imports
from tensorflow.keras.layers import (
    Input,
    LSTM,
    Dense,
    Embedding,
    Concatenate,
    Bidirectional,
    Reshape,
    Lambda,
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import to_categorical
# Necessary imports
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
import bleach
import emoji
import string
import pandas as pd
import re
import emoji
import nltk
import bleach
from nltk.corpus import stopwords
# Download necessary NLTK data
nltk.download('stopwords')
nltk.download('punkt')
# Download NLTK components
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('stopwords')
nltk.download('vader_lexicon')

[nltk_data] Error loading stopwords: <urlopen error [Errno 11001]
[nltk_data]     getaddrinfo failed>
[nltk_data] Error loading punkt: <urlopen error [Errno 11001]
[nltk_data]     getaddrinfo failed>
[nltk_data] Error loading punkt: <urlopen error [Errno 11001]
[nltk_data]     getaddrinfo failed>
[nltk_data] Error loading wordnet: <urlopen error [Errno 11001]
[nltk_data]     getaddrinfo failed>
[nltk_data] Error loading stopwords: <urlopen error [Errno 11001]
[nltk_data]     getaddrinfo failed>
[nltk_data] Error loading vader_lexicon: <urlopen error [Errno 11001]
[nltk_data]     getaddrinfo failed>


False

In [4]:
import pandas as pd
import re
import bleach
import emoji
import nltk
from nltk.corpus import stopwords

data_paths = {
    "genuine_accounts": r"D:\RPP\CRESI-DATASET\cresci-2017.csv\datasets_full.csv\genuine_accounts.csv\genuine_accounts.csv\users.csv",
    "social_spambots_1": r"D:\RPP\CRESI-DATASET\cresci-2017.csv\datasets_full.csv\social_spambots_1.csv\social_spambots_1.csv\users.csv",
    "social_spambots_3": r"D:\RPP\CRESI-DATASET\cresci-2017.csv\datasets_full.csv\social_spambots_3.csv\social_spambots_3.csv\users.csv",
    "traditional_spambots_1": r"D:\RPP\CRESI-DATASET\cresci-2017.csv\datasets_full.csv\traditional_spambots_1.csv\traditional_spambots_1.csv\users.csv",
    "traditional_spambots_2": r"D:\RPP\CRESI-DATASET\cresci-2017.csv\datasets_full.csv\traditional_spambots_2.csv\traditional_spambots_2.csv\users.csv",
    "traditional_spambots_3": r"D:\RPP\CRESI-DATASET\cresci-2017.csv\datasets_full.csv\traditional_spambots_3.csv\traditional_spambots_3.csv\users.csv",
    "traditional_spambots_4": r"D:\RPP\CRESI-DATASET\cresci-2017.csv\datasets_full.csv\traditional_spambots_4.csv\traditional_spambots_4.csv\users.csv" 
}

def preprocess_text_data(input_path, output_path):
    # Read the CSV file
    df = pd.read_csv(input_path, low_memory=False)

    # Lowercase text
    df["description"] = df["description"].str.lower()

    # Remove punctuation and special characters
    def clean_text(text):
        if pd.isna(text):  # Check if the value is NaN
            return ""  # Replace NaN with an empty string
        text = re.sub(r"[^a-zA-Z0-9\s]", "", text)
        return text

    df["description"] = df["description"].apply(clean_text)

    # Removal of stop words
    stop_words = stopwords.words("english")

    def remove_stop_words(text):
        words = [word for word in text.split() if word not in stop_words]
        return " ".join(words)

    df["description"] = df["description"].apply(remove_stop_words)
    # Removal of HTML tags from the text
    def remove_html_tags(text):
        return bleach.clean(text, tags=[], strip=True)

    df["description"] = df["description"].apply(remove_html_tags)

    # Handle negations
    def handle_negations(text):
        negations = {"no", "not", "never", "none", "can't", "couldn't", "didn't", "doesn't", "don't", "isn't",
                     "wasn't", "won't"}
        words = text.split()
        negated_words = [word for word in words if word in negations]
        for word in negated_words:
            index = words.index(word)
            if index < len(words) - 1:
                words[index + 1] = f"NOT_{words[index + 1]}"
        return " ".join(words)

    df["description"] = df["description"].apply(handle_negations)

    # Demojize the text
    def convert_emojis(text):
        return emoji.demojize(text)

    df["description"] = df["description"].apply(convert_emojis)

    # Perform word segmentation
    def perform_word_segmentation(text):
        tokens = nltk.word_tokenize(text)
        segmented_words = ' '.join(tokens)
        return segmented_words
    df['segmented_words'] = df["description"].apply(perform_word_segmentation)
    df.to_csv(output_path, index=False)


In [5]:
    # Save preprocessed data to a new file
input_path = r"D:\\RPP\\COMBINEDUSERS\\usercm.csv"
output_path = r"D:\\RPP\\COMBINEDUSERS\\usercm_pre.csv"
preprocess_text_data(input_path, output_path)

In [6]:
import pandas as pd

# Concatenate CSV files into DataFrame
df = pd.DataFrame()
for label, path in data_paths.items():
    try:
        temp_df = pd.read_csv(path, encoding="ISO-8859-1", low_memory=False)
        temp_df["label"] = 0 if label == "genuine_accounts" else 1
        df = pd.concat([df, temp_df], ignore_index=True)
    except pd.errors.ParserError:
        print(f"Error reading file: {path}. Trying different encodings...")
        try:
            temp_df = pd.read_csv(path, encoding="utf-8")
            temp_df.dropna(inplace=True)
            temp_df["label"] = 0 if label == "genuine_accounts" else 1
            df = pd.concat([df, temp_df], ignore_index=True)
        except Exception as e:
            print(f"Failed to read file: {path}. Error: {e}")

# Save concatenated DataFrame to CSV
df.to_csv("D:\\RPP\\COMBINEDUSERS\\usercm.csv", index=False)

# Preprocess the concatenated DataFrame
preprocess_text_data("D:\\RPP\\COMBINEDUSERS\\usercm.csv", "D:\\RPP\\COMBINEDUSERS\\usercm_pre.csv")

# Load preprocessed data from CSV
df_preprocessed = pd.read_csv("D:\\RPP\\COMBINEDUSERS\\usercm_pre.csv")

# Display first few rows of preprocessed DataFrame
print(df_preprocessed.columns)


Index(['id', 'name', 'screen_name', 'statuses_count', 'followers_count',
       'friends_count', 'favourites_count', 'listed_count', 'url', 'lang',
       'time_zone', 'location', 'default_profile', 'default_profile_image',
       'geo_enabled', 'profile_image_url', 'profile_banner_url',
       'profile_use_background_image', 'profile_background_image_url_https',
       'profile_text_color', 'profile_image_url_https',
       'profile_sidebar_border_color', 'profile_background_tile',
       'profile_sidebar_fill_color', 'profile_background_image_url',
       'profile_background_color', 'profile_link_color', 'utc_offset',
       'is_translator', 'follow_request_sent', 'protected', 'verified',
       'notifications', 'description', 'contributors_enabled', 'following',
       'created_at', 'timestamp', 'crawled_at', 'updated', 'test_set_1',
       'test_set_2', 'label', 'segmented_words'],
      dtype='object')


In [8]:
max_len = df_preprocessed["segmented_words"].str.len().max()
max_len

249.0

In [9]:
df_preprocessed['segmented_words'].head(5)

0                        15years ago xlines24
1                                          de
2                           let see best move
3    20 menna farida nyc 80s actually dragana
4                               cosmetologist
Name: segmented_words, dtype: object

In [10]:
df_preprocessed["segmented_words"].dtype

dtype('O')

In [11]:
max_len = df_preprocessed['description'].apply(lambda x: len(str(x).split())).max()
print("Maximum length of a description:", max_len)

Maximum length of a description: 30


In [12]:
# Function for loading the glove_embedding model.
def load_glove_model(glove_file):
    glove_model = {}
    with open(glove_file, 'r', encoding='utf-8') as file:
        for line in file:
            elements = line.split()
            word = elements[0]
            coefs = np.array(elements[1:], dtype='object')
            glove_model[word] = coefs
    print(f"{len(glove_model)} words loaded!")
    return glove_model

In [13]:
glove_model = load_glove_model("D:\\RPP\\WORD EMBEDING\\glove.twitter.27B.50d.txt")

1193515 words loaded!


In [14]:
def text_to_embeddings(segmented_words, glove_model, embedding_dim=50):
  words = segmented_words.split()
  embeddings = []
  for word in words:
    if word in glove_model:
      embeddings.append(glove_model[word])
    else:
      embeddings.append(np.zeros(embedding_dim))  # Handle unknown words (zero vector)
  return np.array(embeddings, dtype='float32')  # Adjust to 'float32' for efficiency

In [15]:
# Function for applying the glove_model.
def apply_glove_embeddings(df, text_columns, glove_model):
    new_df = pd.DataFrame()
    for column in text_columns:
        new_column_name = f'{column}_glove_embeddings'
        new_df[new_column_name] = df[column].apply(lambda x: text_to_embeddings(str(x), glove_model))
    return new_df

In [16]:
# Apply GloVe embeddings to the DataFrame
text_columns_to_embed = ['segmented_words']
new_df_with_embeddings = apply_glove_embeddings(df_preprocessed, text_columns_to_embed, glove_model)
df = pd.concat([df_preprocessed, new_df_with_embeddings], axis=1)

In [17]:
df.columns

Index(['id', 'name', 'screen_name', 'statuses_count', 'followers_count',
       'friends_count', 'favourites_count', 'listed_count', 'url', 'lang',
       'time_zone', 'location', 'default_profile', 'default_profile_image',
       'geo_enabled', 'profile_image_url', 'profile_banner_url',
       'profile_use_background_image', 'profile_background_image_url_https',
       'profile_text_color', 'profile_image_url_https',
       'profile_sidebar_border_color', 'profile_background_tile',
       'profile_sidebar_fill_color', 'profile_background_image_url',
       'profile_background_color', 'profile_link_color', 'utc_offset',
       'is_translator', 'follow_request_sent', 'protected', 'verified',
       'notifications', 'description', 'contributors_enabled', 'following',
       'created_at', 'timestamp', 'crawled_at', 'updated', 'test_set_1',
       'test_set_2', 'label', 'segmented_words',
       'segmented_words_glove_embeddings'],
      dtype='object')

In [18]:
metadata_features = ['listed_count', 'protected', 'id', 'statuses_count', 'followers_count',
                      'friends_count', 'favourites_count','verified']

# Fill NaN values with zeros and convert to int64 (exclude label and image url)
df.loc[:, metadata_features] = df[metadata_features].fillna(0).astype('int32')


# Normalize metadata features
scaler = StandardScaler()
df[metadata_features[:-1]] = scaler.fit_transform(df[metadata_features[:-1]])

In [19]:
df[metadata_features]

,listed_count,protected,id,statuses_count,followers_count,friends_count,favourites_count,verified
0,-0.054572,-0.102103,1.760997,-0.316358,-0.038432,-0.165055,-0.232793,0.0
1,-0.043695,-0.102103,-2.197162,-0.295424,-0.035139,-0.122908,0.220113,0.0
2,-0.057291,-0.102103,0.312942,-0.356364,-0.039566,-0.207752,-0.120391,0.0
3,0.217368,-0.102103,0.191507,8.386551,0.016639,0.013722,7.102534,0.0
4,-0.057291,-0.102103,-1.564445,-0.407162,-0.043481,-0.234747,-0.264559,0.0
...,...,...,...,...,...,...,...,...
7555,-0.024659,-0.102103,0.941688,-0.408939,-0.039485,-0.214914,-0.265170,0.0
7556,0.027010,-0.102103,0.975264,-0.401268,-0.035085,-0.189020,-0.265170,0.0
7557,-0.035536,-0.102103,0.988403,-0.409979,-0.029740,-0.173869,-0.265170,0.0
7558,-0.054572,-0.102103,1.006432,-0.410630,-0.037379,-0.200865,-0.265170,0.0


In [20]:
embedding_dim = 50
max_len = 26
lstm_units = 100
dropout_rate = 0.5
patience = 3

X_semantics = pad_sequences(df['segmented_words_glove_embeddings'].apply(np.vstack), maxlen=max_len, padding='post')

# Assuming 'label' is the target variable
y = df['label'].values


metadata_features_df = df[metadata_features]  

X_semantics_train, X_semantics_test, metadata_train, metadata_test, y_train, y_test = train_test_split(
    X_semantics, metadata_features_df, y, test_size=0.2, random_state=42
)

In [23]:
# Import libraries
from tensorflow.keras.layers import Input, Bidirectional, LSTM, Lambda, Dense, Concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import binary_crossentropy
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import Callback  
from sklearn.metrics import roc_auc_score

# Define custom AUC callback class
class AUCcallback(Callback):
  def __init__(self):
    self.auc_values = []

  def on_epoch_end(self, epoch, logs=None):
    y_pred = self.model.predict([X_semantics_test, metadata_test])
    y_pred = (y_pred > 0.5).astype(int)  
    auc = roc_auc_score(y_test, y_pred)
    self.auc_values.append(auc)
    print(f"AUC Epoch {epoch+1}: {auc:.4f}")

# Define the neural network architecture with regularizers
def create_model(embedding_dim, max_len, lstm_units):
    semantics_input = Input(shape=(max_len, embedding_dim))
    lstm_output = semantics_input
    for _ in range(5):
        lstm_output = Bidirectional(LSTM(lstm_units, return_sequences=True, kernel_regularizer=l2(0.08)))(lstm_output)

    # Take the last timestep output from the final LSTM layer
    lstm_output = Lambda(lambda x: x[:, -1, :])(lstm_output)
    metadata_input = Input(shape=(metadata_features_df.shape[1],))
    # Apply L2 regularization to Dense layers
    metadata_output = Dense(128, activation='relu', kernel_regularizer=l2(0.08))(metadata_input)
    combined_input = Concatenate()([lstm_output, metadata_output])
    output = Dense(1, activation='sigmoid', kernel_regularizer=l2(0.10))(combined_input)

    model = Model(inputs=[semantics_input, metadata_input], outputs=output)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

# Create the model instance
model = create_model(embedding_dim, max_len, lstm_units)

# Create the AUC callback instance
auc_callback = AUCcallback()

# Train the model
model.fit([X_semantics_train, metadata_train], y_train, batch_size=32, epochs=10,
          validation_data=([X_semantics_test, metadata_test], y_test), callbacks=[auc_callback])

# Evaluate the model
_, accuracy = model.evaluate([X_semantics_test, metadata_test], y_test)
print('Accuracy: %.2f' % (accuracy * 100))


Epoch 1/10
48/48 ━━━━━━━━━━━━━━━━━━━━ 7s 92ms/steptep - accu
AUC Epoch 1: 0.5237
189/189 ━━━━━━━━━━━━━━━━━━━━ 52s 175ms/step - accuracy: 0.7344 - loss: 72.3643 - val_accuracy: 0.5470 - val_loss: 0.8595
Epoch 2/10
48/48 ━━━━━━━━━━━━━━━━━━━━ 2s 38ms/stepep - accur
AUC Epoch 2: 0.5908
189/189 ━━━━━━━━━━━━━━━━━━━━ 21s 113ms/step - accuracy: 0.6178 - loss: 0.7819 - val_accuracy: 0.6104 - val_loss: 0.7152
Epoch 3/10
48/48 ━━━━━━━━━━━━━━━━━━━━ 2s 38ms/stepep - accura
AUC Epoch 3: 0.5938
189/189 ━━━━━━━━━━━━━━━━━━━━ 21s 113ms/step - accuracy: 0.6110 - loss: 0.7083 - val_accuracy: 0.6131 - val_loss: 0.7002
Epoch 4/10
48/48 ━━━━━━━━━━━━━━━━━━━━ 2s 41ms/stepep - acc
AUC Epoch 4: 0.5867
189/189 ━━━━━━━━━━━━━━━━━━━━ 22s 117ms/step - accuracy: 0.6164 - loss: 0.6962 - val_accuracy: 0.6065 - val_loss: 0.6948
Epoch 5/10
48/48 ━━━━━━━━━━━━━━━━━━━━ 2s 39ms/stepep - accura
AUC Epoch 5: 0.5747
189/189 ━━━━━━━━━━━━━━━━━━━━ 22s 114ms/step - accuracy: 0.6032 - loss: 0.6919 - val_accuracy: 0.5952 - val_loss: 0

In [28]:
full_features=model.predict([X_semantics_test, metadata_test],)
full_features

48/48 ━━━━━━━━━━━━━━━━━━━━ 2s 31ms/step


array([[0.54355663],
       [0.53981185],
       [0.5432897 ],
       ...,
       [0.5441797 ],
       [0.54342544],
       [0.5475929 ]], dtype=float32)

In [24]:
model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 26, 50)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_5     │ (None, 26, 200)   │    120,800 │ input_layer_2[0]… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_6     │ (None, 26, 200)   │    240,800 │ bidirectional_5[… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_7     │ (None, 26, 200)   │    240,800 │ bidirectional_6[… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_8     │ (None, 26, 200)   │    240,800 │ bidirectional_7[… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_9     │ (None, 26, 200)   │    240,800 │ bidirectional_8[… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_3       │ (None, 8)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_1 (Lambda)   │ (None, 200)       │          0 │ bidirectional_9[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 128)       │      1,152 │ input_layer_3[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 328)       │          0 │ lambda_1[0][0],   │
│ (Concatenate)       │                   │            │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 1)         │        329 │ concatenate_1[0]… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 3,256,445 (12.42 MB)

 Trainable params: 1,085,481 (4.14 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2,170,964 (8.28 MB)